In [0]:
LANDING_PATH="/Volumes/workspace/training/transactions/landing/"
CHECKPOINT_PATH="/Volumes/workspace/training/transactions/checkpoints/"
BRONZE_TABLE="workspace.training.transactions_bronze"

In [0]:
from pyspark.sql.functions import current_timestamp, col
df_bronze=(spark.readStream.format("cloudFiles")
           .option("cloudFiles.format", "csv")
           .option("cloudFiles.schemaLocation", CHECKPOINT_PATH+"schema/")
           .option("cloudFiles.inferColumnTypes", "true")
           .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
           .option("header", "true")
           .option("quote", '"')
           .option("escape", '"')
           .option("mode", "PERMISSIVE")
           .load(LANDING_PATH)
           .withColumn("_source_file", col("_metadata.file_path"))
           .withColumn("_ingested_at", current_timestamp())
)
(df_bronze.writeStream.format("delta")
                      .outputMode("append")
                      .option("checkpointLocation", CHECKPOINT_PATH)
                      .option("mergeSchema", "true")
                      .trigger(availableNow=True)
                      .toTable(BRONZE_TABLE)
                      .awaitTermination())

spark.table(BRONZE_TABLE).display()   

SILVER LAYER

In [0]:
SILVER_TABLE = "workspace.training.transactions_silver"

In [0]:
from pyspark.sql.functions import to_date, col, from_json, expr, row_number
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.window import Window

product_schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True)
])

payment_schema = StructType([
    StructField("payment_mode", StringType(), True),
    StructField("channel", StringType(), True)
])

window_spec = Window.partitionBy("tno", "store_no").orderBy(
    col("t_date").desc(), 
    col("unit_price").desc()
)

df_silver = (spark.table(BRONZE_TABLE)

    .withColumn("product_info", from_json(col("product_info"), product_schema))
    .withColumn("payment_info", from_json(col("payment_info"), payment_schema))

    .withColumn("product_id", col("product_info.product_id").cast("string"))
    .withColumn("product_name", col("product_info.product_name").cast("string"))
    .withColumn("category", col("product_info.category").cast("string"))
    .withColumn("payment_mode", col("payment_info.payment_mode").cast("string"))
    .withColumn("channel", col("payment_info.channel").cast("string"))
    .drop("product_info", "payment_info")


    .withColumn("t_date", expr("try_to_date(t_date , 'yyyy-MM-dd')"))
    .withColumn("quantity", expr("try_cast(quantity as int)"))
    .withColumn("unit_price", expr("try_cast(unit_price as decimal(12,2))"))
    .withColumn("discount_pct", expr("try_cast(discount_pct as decimal(5,4))"))
    .withColumn("tax_rate", expr("try_cast(tax_rate as decimal(5,4))"))
    .withColumn("net_amount", expr("try_cast(net_amount as decimal(14,2))"))

    .filter(col("tno").isNotNull() & col("store_no").isNotNull() & col("t_date").isNotNull())
    .filter(col("quantity").isNotNull() & (col("quantity") > 0))
    .filter(col("net_amount").isNotNull() & (col("net_amount") > 0))

    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)

    .orderBy("tno")
)

(df_silver.limit(0).write.format("delta")
    .mode('ignore')
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE))
df_silver.display()

In [0]:
df_silver.createOrReplaceTempView("df_silver_view")

spark.sql(f"""
    MERGE INTO {SILVER_TABLE} AS tgt
    USING df_silver_view AS src
    ON tgt.tno = src.tno AND tgt.store_no = src.store_no AND tgt.t_date = src.t_date
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

spark.table(SILVER_TABLE).display()

In [0]:
spark.sql(f"DESCRIBE HISTORY {SILVER_TABLE}").display()

In [0]:
from pyspark.sql.functions import count

# Counts all rows and aliases the column name
df_silver.select(count("*").alias("total")).show()